In [1]:
import os
from utils_extraction import extract_body, tokenize, clean_tokens, decode
from utils_extraction import chunk_tokens, flatten_token_chunks
from utils_extraction import extract_few_shot_examples
from utils_extraction import select_few_shot 
from utils_extraction import merge_tokens_with_auto_labels, add_style_and_parent_to_auto_labels, compare_html_allow_auto_labels
from models import GPTAssistant
from process_chunks import process_chunks
from utils_extraction import is_manual_label_tag, is_auto_label_tag
from utils_extraction.html_utils import clean_html_formatting

In [2]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 15  # Number of few-shot examples to use

#### Define the text to process, and where to save it. Define the text for few shot

In [3]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "2019SCC65"
round = "ronde_1"
anno = "llm"
version = "v1.0_2"
html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.htm"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"

os.makedirs(output_dir, exist_ok=True)

# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "1999CanLII7320_annotated"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}_{fs_anno}_{fs_version}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_1\plain_html_arbre_balise\2019SCC65.htm
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_EG_v1.html


### Process The HTML Content

In [4]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)


   ⚠ Warning: stop_bookmark_separation=True but bookmark not found
   ✓ Chunked tokens into 285 chunks (>= 500 tokens each)


In [5]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 23066
   ✓ Splitting: 23066 tokens before, 31001 tokens after
   ✓ Chunked tokens into 36 chunks (>= 500 tokens each)
   ✓ Chunked tokens into 62 chunks (>= 500 tokens each)
   ✓ Total chunks: 36 before + 62 after = 98


In [6]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [7]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 36 few-shot examples from chunks
   ✓ Selected 15 few-shot examples for processing.


In [8]:
selected_few_shot_examples

[(" General Accident Assurance Company et al. v. Chrusz et al. Chrusz et al. v. General Accident Assurance Company et al. [Indexed as: General Accident Assurance Co. v. Chrusz] 45 O.R. (3d) 321 [1999] O.J. No. 3291 Docket No. C29463 Court of Appeal for Ontario Carthy, Doherty and Rosenberg JJ.A. September 14, 1999 Civil procedure -- Discovery -- Privilege -- Solicitor-client privilege -- Litigation privilege -- Common interest privilege -- Hotel destroyed by fire -- Insurance adjuster investigating fire -- Suspicion of arson -- Adjuster directed to provide reports directly to lawyer retained by insurer -- Insurer later making partial payments of insurance -- Subsequently, dismissed employee alleging that insured's claim fraudulent -- Insured's lawyer providing dismissed employee with copy of transcript of his statement -- Insurer suing insured -- Insured making counterclaim and joining employee -- Adjuster's reports before allegation of fraud not privileged -- Adjuster's reports after 

In [9]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name, temperature=1)

In [10]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction_cot.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)


   ✓ Processing 285 chunks with LLM...
   ✓ Using 15 few-shot examples


Processing chunks:  18%|█▊        | 51/285 [05:22<23:53,  6.13s/it]

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks:  20%|██        | 57/285 [06:00<23:28,  6.18s/it]

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks:  88%|████████▊ | 252/285 [26:10<03:20,  6.07s/it] 

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks: 100%|██████████| 285/285 [29:15<00:00,  6.16s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\history_2019SCC65.json

   ✓ Processing completed:
      - Total chunks: 285
      - Successful: 285
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65\processed_chunks_2019SCC65.json


In [11]:
#write the processed chuncks in a json file for later use in the annotation interface

import json 
with open(f"{output_dir}\\processed_chunks_v2.json", "w") as f:
    json.dump(processed_chunks, f)

## Post Processing

In [ ]:
# Read the processed_chuncks.json file to verify it was written correctly
import json
with open(f"{output_dir}\\processed_chunks.json", "r") as f:
    processed_chunks = json.load(f)

In [12]:
def is_auto_label_tag(tok: str) -> bool:
    """Return 1 if token is an opening, 2 if it is a closing auto_label tag, 0 otherwise"""
    if not is_tag_token(tok):
        return 0
    # Accept variations with attributes on opening tag
    if tok.lower().startswith('<auto_label'):
        return 1
    if tok.lower().startswith('</auto_label'):
        return 2
    return 0

def is_manual_label_tag(tok):
    """Return 1 if token is an opening, 2 if it is a closing manual_label tag, 0 otherwise"""
    if not is_tag_token(tok):
        return 0
    # Accept variations with attributes on opening tag
    if tok.lower().startswith('<manual_label'):
        return 1
    if tok.lower().startswith('</manual_label'):
        return 2
    return 0
def is_tag_token(tok: str) -> bool:
    """Return True if token looks like an HTML tag (e.g., <...>)."""
    return len(tok) >= 3 and tok[0] == '<' and tok[-1] == '>'

def is_fmt_tag(tok: str, fmt_tags: set) -> bool:
    """Check if token is a formatting tag (opening or closing)."""
    if not is_tag_token(tok):
        return False
    
    tag_name = get_tag_name(tok)
    return tag_name in fmt_tags

def is_opening_tag(tok: str) -> bool:
    """Check if token is an opening tag (not closing)."""
    return is_tag_token(tok) and not tok.startswith('</')


def is_closing_tag(tok: str) -> bool:
    """Check if token is a closing tag."""
    return is_tag_token(tok) and tok.startswith('</')

def get_tag_name(tok: str) -> str:
    """
    Extract the tag name from a token.
    Examples:
        '<i>' -> 'i'
        '</i>' -> 'i'
        '<span class="test">' -> 'span'
        '<auto_label labelname="decision">' -> 'auto_label'
    """
    if not is_tag_token(tok):
        return ""
    
    # Remove < and >
    content = tok[1:-1]
    
    # Remove leading / for closing tags
    if content.startswith('/'):
        content = content[1:]
    
    # Split on whitespace to get just the tag name (handles attributes)
    tag_name = content.split()[0] if content else ""
    
    return tag_name

In [13]:
def correct_tokens_brackets(tokens, fmt_tags = {"i", "b", "strong", "u", "em", "mark", "span"}):
    """"
    Fix formatting tag nesting issues caused by auto_label insertion.
    
    The function ensures that formatting tags don't cross auto_label boundaries.
    When a closing </auto_label> is encountered, all open formatting tags are closed
    before it, then reopened after it.
    
    CASE 1: Direct nesting issues
        <i><auto_label>text</i> more</auto_label> 
        -> <auto_label><i>text</i> more</auto_label>
    
    CASE 2: Indirect nesting issues
        <span>text <auto_label>more</span> text</auto_label>
        -> <span>text</span><auto_label><span>more</span> text</auto_label>


    The function process the folowing problem caused by auto_label insertion:
    CASE 1 : ERROR DETECTED, and one of the tag is directly outside or inside auto_label whitout anyspace : it as to get in or get out
        <i><auto_label labelname="decision">Some text</i> Some text </auto_label> -> <auto_label labelname="decision"><i>Some text</i> Some text </auto_label> 
        <i>Some text <auto_label labelname="decision"></i> Some text </auto_label> -> <i>Some text </i><auto_label labelname="decision"> Some text </auto_label> 
        Some text <auto_label labelname="decision">Some text <i> Some text </auto_label></i> -> Some text <auto_label labelname="decision">Some text <i> Some text </i></auto_label> 

    CASE 2 : ERROR DETECTED but no label is directly next to a autotag :
         <span class=""> Some text <auto_label labelname="decision">Some text </span> Some text </auto_label> -> <span class=""> Some text </span><auto_label labelname="decision"><span class=""> Some text </span> Some text </auto_label>
         <auto_label labelname="decision">Some text <span class=""> Some text </auto_label> Some text </span> -> <auto_label labelname="decision">Some text <span class=""> Some text </span></auto_label><span class="" Some text </span>
    """

    corrected = []
    fmt_stack = []  # Stack of (tag_name, full_opening_token)

    for tok in tokens:
        
        # Handle auto_label opening
        auto_label_type = is_auto_label_tag(tok)
        if auto_label_type == 1:  # Opening <auto_label>
            # Close all open formatting tags BEFORE opening auto_label
            to_reopen = []
            
            while fmt_stack:
                tag_name, open_tok = fmt_stack.pop()
                corrected.append(f"</{tag_name}>")
                to_reopen.append((tag_name, open_tok))
            
            # Now add the opening auto_label
            corrected.append(tok)
            
            # Reopen formatting tags INSIDE auto_label
            for tag_name, open_tok in reversed(to_reopen):
                corrected.append(open_tok)
                fmt_stack.append((tag_name, open_tok))
            
            continue
        
        # Handle auto_label closing
        if auto_label_type == 2:  # Closing </auto_label>
            # Close all open formatting tags BEFORE closing auto_label
            to_reopen = []
            
            while fmt_stack:
                tag_name, open_tok = fmt_stack.pop()
                corrected.append(f"</{tag_name}>")
                to_reopen.append((tag_name, open_tok))
            
            # Now add the closing auto_label
            corrected.append(tok)
            
            # Reopen formatting tags AFTER auto_label
            for tag_name, open_tok in reversed(to_reopen):
                corrected.append(open_tok)
                fmt_stack.append((tag_name, open_tok))
            
            continue
        
        # Handle opening formatting tags
        if is_opening_tag(tok):
            tag_name = get_tag_name(tok)
            if tag_name in fmt_tags:
                fmt_stack.append((tag_name, tok))
                corrected.append(tok)
                continue
        
        # Handle closing formatting tags
        if is_closing_tag(tok):
            tag_name = get_tag_name(tok)
            if tag_name in fmt_tags:
                # Remove matching opening tag from stack (search from end)
                for i in range(len(fmt_stack) - 1, -1, -1):
                    if fmt_stack[i][0] == tag_name:
                        fmt_stack.pop(i)
                        break
                
                corrected.append(tok)
                continue
        
        # All other tokens (text, non-fmt tags, etc.)
        corrected.append(tok)

    return corrected


def check_tokens_brackets(tokens, fmt_tags={"i", "b", "strong", "u", "em", "mark", "span"}, ctx=10):
    """
    Verify that tags are properly nested and matched.
    
    Returns:
        tuple: (ok, message, position, context)
            - ok (bool): True if brackets are coherent
            - message (str): Description of the result or error
            - position (int or None): Index of error token if any
            - context (str or None): Surrounding tokens for debugging
    """
    stack = []  # Stack of (tag_name, index, full_token)

    def context_at(i):
        """Get surrounding context for error reporting."""
        start = max(0, i - ctx)
        end = min(len(tokens), i + ctx + 1)
        snippet = tokens[start:end]
        return " ".join(snippet)

    for idx, tok in enumerate(tokens):
        
        # Skip non-tag tokens
        if not is_tag_token(tok):
            continue
        
        # Handle auto_label opening
        auto_label_type = is_auto_label_tag(tok)
        if auto_label_type == 1:  # Opening auto_label
            stack.append(("auto_label", idx, tok))
            continue
        
        # Handle auto_label closing
        if auto_label_type == 2:  # Closing auto_label
            if not stack:
                return False, "Closing </auto_label> with empty stack", idx, context_at(idx)
            
            open_name, open_idx, open_tok = stack.pop()
            
            if open_name != "auto_label":
                msg = f"Mismatched tags: opened {open_tok} at {open_idx}, closed </auto_label>"
                return False, msg, idx, context_at(idx)
            
            continue
        
        # Handle opening formatting tags
        if is_opening_tag(tok) and is_fmt_tag(tok, fmt_tags):
            tag_name = get_tag_name(tok)
            stack.append((tag_name, idx, tok))
            continue
        
        # Handle closing formatting tags
        if is_closing_tag(tok) and is_fmt_tag(tok, fmt_tags):
            tag_name = get_tag_name(tok)
            
            if not stack:
                return False, f"Closing tag {tok} with empty stack", idx, context_at(idx)
            
            open_name, open_idx, open_tok = stack.pop()
            
            if open_name != tag_name:
                msg = f"Mismatched tags: opened {open_tok} at {open_idx}, closed {tok}"
                return False, msg, idx, context_at(idx)
            
            continue

    # Check for unclosed tags
    if stack:
        name, idx, tok = stack[-1]
        return False, f"Unclosed tag {tok}", idx, context_at(idx)

    return True, "Brackets are coherent", None, None



    

In [14]:
# ---------- Merge all tokens ----------
processed_tokens_flat = flatten_token_chunks(processed_chunks)


original_tokens = tokenize(html_content)
processed_html_content_tokens = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)






   ✓ Flattened 285 chunks into 145710 tokens


In [ ]:
# Correcte the oppening closing tag caused by auto_label insertion
#processed_html = decode(processed_html_content_tokens)

#print(f"\nMerged HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
#processed_html_content = decode(add_style_and_parent_to_auto_labels(processed_html))


# ---------- Compare with original HTML (ignoring auto_label tags) ----------
#comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)

In [ ]:
#print(check_tokens_brackets(processed_html_content_tokens))

(False, 'Mismatched tags: opened <auto_label labelname="decision"> at 909, closed </span>', 911, '\xa0 </span> </p> \n <br clear="ALL"/> \n <p class="MsoNormal" style="margin-bottom:.5in;text-align:justify"> <span class="SCCAppellantForRunningHeadChar"> <auto_label labelname="decision"> canada </span> <span style="font-variant:small-caps">   </span> <i> v . </i> <span style="font-variant:small-caps">   vavilov')


In [15]:
corrected_processed_html_content_tokens = correct_tokens_brackets(tokens=processed_html_content_tokens)
print(check_tokens_brackets(corrected_processed_html_content_tokens))


(True, 'Brackets are coherent', None, None)


In [16]:
# Correcte the oppening closing tag caused by auto_label insertion
processed_html = decode(corrected_processed_html_content_tokens)
processed_html_cleaned = clean_html_formatting(processed_html)
print(f"\nMerged HTML length: {len(processed_html)}")


Merged HTML length: 652675


In [17]:
# ---------- Add style and parent to auto_label tags ----------
processed_html_content = add_style_and_parent_to_auto_labels(processed_html_cleaned)


# ---------- Compare with original HTML (ignoring auto_label tags) ----------
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


In [18]:
# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2019SCC65


In [ ]:
print(processed_html_content)

<html><!-- HTMLLabelizer
{
  "labeltree": {
    "legislation": {
        "color": "#76CEDE",
        "sublabels": {
            "title": {
                "color": "#93c47d",
                "sublabels": {},
                "attributes": {
                    "titletype": {
                        "type": "dropdown",
                        "options": [
                            "official",
                            "alias"
                        ],
                        "default": "official",
                        "groupRole": "regular"
                    }
                }
            },
            "citation": {
                "color": "#ff5733",
                "sublabels": {},
                "attributes": {}
            },
            "fragment": {
                "color": "#8e7cc3",
                "sublabels": {},
                "attributes": {
                    "fragmentid": {
                        "type": "string",
                        "default": "",
     